In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/customers.csv")

print(df.info())
print(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CustomerID     10 non-null     int64  
 1   Age            9 non-null      float64
 2   City           10 non-null     str    
 3   AnnualIncome   9 non-null      float64
 4   SpendingScore  10 non-null     int64  
 5   Membership     10 non-null     str    
dtypes: float64(2), int64(2), str(2)
memory usage: 612.0 bytes
None
CustomerID       0
Age              1
City             0
AnnualIncome     1
SpendingScore    0
Membership       0
dtype: int64
Duplicate rows: 0


In [2]:
print(df.isna().sum())
print(df.isna().mean().mul(100).round(2))

CustomerID       0
Age              1
City             0
AnnualIncome     1
SpendingScore    0
Membership       0
dtype: int64
CustomerID        0.0
Age              10.0
City              0.0
AnnualIncome     10.0
SpendingScore     0.0
Membership        0.0
dtype: float64


In [6]:
# Remove rows where Age is missing
df_without_missing_age = df.dropna(subset=["Age"])
df_without_missing_age


,CustomerID,Age,City,AnnualIncome,SpendingScore,Membership
0,101,22.0,Hyderabad,350000.0,75,Gold
1,102,28.0,Bengaluru,600000.0,60,Silver
2,103,35.0,Chennai,800000.0,45,Gold
4,105,42.0,Mumbai,1200000.0,35,Gold
5,106,30.0,Bengaluru,NaN,65,Silver
6,107,25.0,Chennai,500000.0,90,Gold
7,108,38.0,Mumbai,950000.0,40,Silver
8,109,29.0,Hyderabad,550000.0,70,Gold
9,110,45.0,Bengaluru,1500000.0,25,Silver


In [7]:
df_clean = df.copy()

df_clean["Age"] = df_clean["Age"].fillna(
    df_clean["Age"].median()
)

df_clean["AnnualIncome"] = df_clean["AnnualIncome"].fillna(
    df_clean["AnnualIncome"].median()
)

print(df_clean.isna().sum())

CustomerID       0
Age              0
City             0
AnnualIncome     0
SpendingScore    0
Membership       0
dtype: int64


In [8]:
print("Duplicate rows:", df_clean.duplicated().sum())

# Display duplicate rows
print(df_clean[df_clean.duplicated(keep=False)])

# Remove complete duplicate rows
df_clean = df_clean.drop_duplicates()

Duplicate rows: 0
Empty DataFrame
Columns: [CustomerID, Age, City, AnnualIncome, SpendingScore, Membership]
Index: []


In [9]:
print(df_clean[df_clean.duplicated(subset=["CustomerID"], keep=False)])

Empty DataFrame
Columns: [CustomerID, Age, City, AnnualIncome, SpendingScore, Membership]
Index: []


In [10]:
df_clean["CustomerID"] = df_clean["CustomerID"].astype("int64")

print(df_clean.dtypes)

CustomerID         int64
Age              float64
City                 str
AnnualIncome     float64
SpendingScore      int64
Membership           str
dtype: object


In [11]:
dates = pd.Series(["2026-01-10", "2026-02-15", "invalid-date"])

parsed_dates = pd.to_datetime(dates, errors="coerce")

print(parsed_dates)

0   2026-01-10
1   2026-02-15
2          NaT
dtype: datetime64[us]


In [13]:
values = pd.Series(["100", "200", "invalid", "400"])

numbers = pd.to_numeric(values, errors="coerce")

print(numbers)

0    100.0
1    200.0
2      NaN
3    400.0
dtype: float64


In [14]:
df_clean["Membership"] = (
    df_clean["Membership"]
    .astype("string")
    .str.strip()
    .str.title()
)

df_clean["City"] = (
    df_clean["City"]
    .astype("string")
    .str.strip()
)

print(df_clean["Membership"].unique())
print(df_clean["City"].unique())

<StringArray>
['Gold', 'Silver']
Length: 2, dtype: string
<StringArray>
['Hyderabad', 'Bengaluru', 'Chennai', 'Mumbai']
Length: 4, dtype: string


In [15]:
invalid_age = df_clean[
    (df_clean["Age"] < 0) |
    (df_clean["Age"] > 120)
]

invalid_income = df_clean[df_clean["AnnualIncome"] < 0]

invalid_score = df_clean[
    ~df_clean["SpendingScore"].between(0, 100)
]

print("Invalid ages:", len(invalid_age))
print("Invalid incomes:", len(invalid_income))
print("Invalid scores:", len(invalid_score))

Invalid ages: 0
Invalid incomes: 0
Invalid scores: 0


In [16]:
def clean_customers(data):
    df = data.copy()

    # Standardize text
    for col in ["City", "Membership"]:
        df[col] = df[col].astype("string").str.strip()

    df["Membership"] = df["Membership"].str.title()

    # Remove duplicate customer IDs only if one row per
    # customer is a valid business rule for this dataset.
    df = df.drop_duplicates(subset=["CustomerID"], keep="first")

    # Fill missing numerical values with medians
    for col in ["Age", "AnnualIncome"]:
        df[col] = df[col].fillna(df[col].median())

    return df


df_clean = clean_customers(df)

print(df_clean.head())
print(df_clean.isna().sum())
print("Remaining duplicates:", df_clean.duplicated().sum())

   CustomerID   Age       City  AnnualIncome  SpendingScore Membership
0         101  22.0  Hyderabad      350000.0             75       Gold
1         102  28.0  Bengaluru      600000.0             60     Silver
2         103  35.0    Chennai      800000.0             45       Gold
3         104  30.0  Hyderabad      450000.0             80     Silver
4         105  42.0     Mumbai     1200000.0             35       Gold
CustomerID       0
Age              0
City             0
AnnualIncome     0
SpendingScore    0
Membership       0
dtype: int64
Remaining duplicates: 0


In [18]:
from pathlib import Path

output_path = Path("../data/processed/customers_clean.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved to:", output_path)

Cleaned dataset saved to: ..\data\processed\customers_clean.csv
